In [1]:
!pip install datasets
!pip install huggingface_hub

In [2]:
import sys
import json
import hashlib
from pathlib import Path

In [3]:
from datasets import load_dataset, Dataset
from huggingface_hub import hf_hub_download

In [4]:
CACHE_DIR = Path.home() / ".cache" / "huggingface" / "datasets"

In [5]:
def load_and_inspect(dataset_name):
    ds = load_dataset(dataset_name, split="train")
    print("Dataset:", dataset_name)
    print("Rows:", len(ds))
    print("Columns:", ds.column_names)
    print("First row:", ds[0])
    return ds

In [6]:
def stream_dataset(dataset_name, max_rows=5):
    ds = load_dataset(
        dataset_name,
        split="train",
        streaming=True
    )
    rows = []
    for example in ds:
        rows.append(example)
        if len(rows) == max_rows:
            break

    return rows

In [8]:
def convert_format(ds, output_dir, name):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    csv_path = output_path / f"{name}.csv"
    json_path = output_path / f"{name}.json"
    parquet_path = output_path / f"{name}.parquet"

    ds.to_csv(str(csv_path))
    ds.to_json(str(json_path))
    ds.to_parquet(str(parquet_path))

    print("Files saved successfully!")

    return {
        "csv": csv_path,
        "json": json_path,
        "parquet": parquet_path
    }

In [9]:
def make_splits(ds):
    first_split = ds.train_test_split(test_size=0.2, seed=42)

    train_ds = first_split["train"]

    second_split = first_split["test"].train_test_split(
        test_size=0.5,
        seed=42
    )

    val_ds = second_split["train"]
    test_ds = second_split["test"]

    return {
        "train": train_ds,
        "val": val_ds,
        "test": test_ds
    }

In [10]:
def download_model_file(repo_id, filename):
    path = hf_hub_download(repo_id=repo_id, filename=filename)

    print("Downloaded:", path)

    return path

In [11]:
def cache_summary():
    if not CACHE_DIR.exists():
        print("No cache found")
        return

    total_size = 0

    for file in CACHE_DIR.rglob("*"):
        if file.is_file():
            total_size += file.stat().st_size

    print("Cache size:", total_size / (1024 * 1024), "MB")

In [12]:
def load_from_parquet(path: str):
    ds = Dataset.from_parquet(path)
    print(f"Loaded {len(ds)} rows from {path}")
    return ds


def load_from_csv(path: str):
    ds = Dataset.from_csv(path)
    print(f"Loaded {len(ds)} rows from {path}")
    return ds


def load_from_json(path: str):
    ds = Dataset.from_json(path)
    print(f"Loaded {len(ds)} rows from {path}")
    return ds

In [13]:
def fingerprint(ds):
    rows = list(ds.select(range(min(100, len(ds)))))

    content = json.dumps(rows, default=str).encode()

    hash_value = hashlib.sha256(content).hexdigest()

    return hash_value[:16]

In [16]:
if __name__ == "__main__":
    print("=" * 60)
    print("Data Management Utility")
    print("=" * 60)

    print("\n--- 1. Load and inspect a dataset ---")
    ds = load_and_inspect("cornell-movie-review-data/rotten_tomatoes")

    print("\n--- 2. Stream a dataset ---")
    rows = stream_dataset("cornell-movie-review-data/rotten_tomatoes", max_rows=3)
    for row in rows:
        print(f"  {row['text'][:80]}...")

    print("\n--- 3. Convert formats ---")
    small_ds = ds.select(range(500))
    paths = convert_format(small_ds, "/tmp/data_utils_demo", "rotten_tomatoes_sample")

    print("\n--- 4. Create train/val/test splits ---")
    splits = make_splits(small_ds)

    print("\n--- 5. Reload from Parquet ---")
    reloaded = load_from_parquet(str(paths["parquet"]))
    print(f"  Columns: {reloaded.column_names}")

    print("\n--- 6. Download a model file ---")
    download_model_file("sentence-transformers/all-MiniLM-L6-v2", "config.json")

    print("\n--- 7. Dataset fingerprint ---")
    fingerprint(ds)

    print("\n--- 8. Cache summary ---")
    cache_summary()

    print("\n" + "=" * 60)
    print("All checks passed. Your data pipeline is ready.")
    print("=" * 60)

Data Management Utility

--- 1. Load and inspect a dataset ---
Dataset: cornell-movie-review-data/rotten_tomatoes
Rows: 8530
Columns: ['text', 'label']
First row: {'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}

--- 2. Stream a dataset ---
  the rock is destined to be the 21st century's new " conan " and that he's going ...
  the gorgeously elaborate continuation of " the lord of the rings " trilogy is so...
  effective but too-tepid biopic...

--- 3. Convert formats ---


Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Files saved successfully!

--- 4. Create train/val/test splits ---

--- 5. Reload from Parquet ---


Generating train split: 0 examples [00:00, ? examples/s]

Loaded 500 rows from /tmp/data_utils_demo/rotten_tomatoes_sample.parquet
  Columns: ['text', 'label']

--- 6. Download a model file ---


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

Downloaded: /root/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config.json

--- 7. Dataset fingerprint ---

--- 8. Cache summary ---
Cache size: 1.355423927307129 MB

All checks passed. Your data pipeline is ready.
